# 01 Retrieval Basics / พื้นฐานการค้นคืนข้อมูล

This notebook teaches the RAG sequence: **ingest -> chunk -> retrieve with keywords -> retrieve with vectors -> cite**. It is safe to run in Google Colab without an API key.

สมุดงานนี้สอนลำดับ RAG: **นำเข้าข้อมูล -> แบ่งข้อความ -> ค้นคืนด้วยคำสำคัญ -> ค้นคืนด้วย vector -> อ้างอิง** ใช้ใน Google Colab ได้โดยไม่ต้องมี API key

In [ ]:
import math
from pprint import pprint

# Eight approved, bilingual readings. In the repository, the full 8 readings are in corpus/.
documents = [
  {'id':'human-centred','title':'Human-centred use of GenAI','title_th':'การใช้ GenAI ที่ยึดมนุษย์เป็นศูนย์กลาง','text':'Human agency is a core consideration when education systems design or adopt generative AI. A school identifies the learning problem first, then decides whether an AI tool helps more than a teacher-led activity or another resource. Learners can inspect the source before trusting an answer, and teachers can change the approved corpus and retrieval rule.'},
  {'id':'data-privacy','title':'Protecting learner data','title_th':'การคุ้มครองข้อมูลของผู้เรียน','text':'Education providers should tell learners what data a GenAI system may collect and how it may use those data. Before adoption, identify prompts, files, account details, device information, and interaction history that a tool receives. In production, permission checks must occur before retrieval so a query cannot reveal another person’s documents.'},
  {'id':'bias-validation','title':'Validate bias and representation','title_th':'ตรวจสอบอคติและความครอบคลุมของข้อมูล','text':'Validation mechanisms should examine bias and whether training data represent diversity. Teams should vary names, languages, access needs, and situations in test questions. A similarity score ranks candidates; it does not prove truth, completeness, or fairness.'},
  {'id':'pedagogical-validation','title':'Pedagogical appropriateness','title_th':'ความเหมาะสมด้านการสอน','text':'Use should be proportionate to learner age, expected outcome, and type of knowledge or problem. If learners need to compare approved policies, retrieval with visible citations can support evaluation. The tool, activity, and assessment should support the same learning outcome.'},
  {'id':'human-agency','title':'Human control and accountability','title_th':'การควบคุมและความรับผิดชอบของมนุษย์','text':'Educators and learners should control GenAI use and remain accountable for accuracy. A RAG assistant should return a grounded answer for focused evidence, request clarification for a broad question, and decline when evidence is absent.'},
  {'id':'co-design','title':'Co-design and evaluation','title_th':'การร่วมออกแบบและประเมินผล','text':'Safe use should be co-designed and piloted and evaluated for effectiveness and long-term impact. A useful pilot states its success condition before it launches. The evidence log records the query, retrieved source, score, outcome, and explanation of a failure.'},
  {'id':'teacher-competencies','title':'Teacher AI competencies','title_th':'สมรรถนะ AI สำหรับครู','text':'Teacher capability includes human-centred mindset, AI ethics, foundations, pedagogy, and professional learning. Teachers choose an approved corpus, decide whether keyword or semantic retrieval suits a question, and check whether citations allow learners to inspect claims.'},
  {'id':'student-competencies','title':'Student AI competencies','title_th':'สมรรถนะ AI สำหรับผู้เรียน','text':'Learners are responsible users and co-creators across human-centred mindset, ethics, techniques, and system design. They inspect a document, chunk it, query it by words and related meaning, then judge whether retrieved evidence supports an answer.'}
]
len(documents)

## Full PDF ingestion / นำเข้า PDF ฉบับเต็ม

The repository includes a 48-page UNESCO source PDF. In Google Colab, run the next cell and upload `corpus/source-pdfs/unesco-genai-guidance.pdf`. The notebook then replaces the short readings with page-level records from the full document. Set `USE_FULL_PDF` to `False` only for the fast fallback demonstration.

Repository มี PDF ของ UNESCO จำนวน 48 หน้า ให้รัน cell ถัดไปและอัปโหลด `corpus/source-pdfs/unesco-genai-guidance.pdf` เพื่อแทนที่ชุดอ่านสั้นด้วยข้อมูลระดับหน้าเอกสารจริง ตั้งค่า `USE_FULL_PDF` เป็น `False` เฉพาะเมื่อต้องการสาธิต fallback อย่างรวดเร็ว

In [ ]:
USE_FULL_PDF = True  # Recommended: parse the real 48-page corpus.

if USE_FULL_PDF:
    !pip -q install pypdf
    from google.colab import files
    from pypdf import PdfReader
    import re
    uploaded = files.upload()
    pdf_path = next(iter(uploaded))
    reader = PdfReader(pdf_path)
    documents = []
    for page_number, page in enumerate(reader.pages, start=1):
        page_text = re.sub(r'\s+', ' ', page.extract_text() or '').strip()
        if len(page_text) >= 120:
            documents.append({
                'id': f'unesco-genai-p{page_number:02d}',
                'title': 'UNESCO Guidance for generative AI in education and research',
                'title_th': 'แนวทาง UNESCO สำหรับ GenAI ในการศึกษาและการวิจัย',
                'page': page_number,
                'source_url': 'https://unesdoc.unesco.org/ark:/48223/pf0000386693',
                'text': page_text
            })
    print(f'Loaded {len(documents)} text pages from {len(reader.pages)} PDF pages.')
else:
    print('Using the compact practice corpus. Set USE_FULL_PDF=True to parse the full source PDF.')

In [ ]:
# Chunking: keep document identity so a later answer can cite its origin.
def chunk_document(document, max_words=180, overlap=30):
    words = document['text'].split()
    step = max_words - overlap
    return [{'chunk_id': f"{document['id']}-c{i:02d}", 'document_id': document['id'], 'page': document.get('page'), 'source_url': document.get('source_url'), 'text': ' '.join(words[i:i+max_words])} for i in range(0, len(words), step)]

chunks = [chunk for document in documents for chunk in chunk_document(document)]
print(f'{len(documents)} documents produced {len(chunks)} chunks.')
pprint(chunks[:3])

## Retrieval 1: keyword matching / การค้นคืนแบบคำสำคัญ

Retrieval begins by selecting evidence. Exact and keyword matching are useful when a learner uses a course code, policy title, or words that appear in a source.

In [ ]:
import re

query = 'How should a school protect learner data when using GenAI?'
def tokens(text):
    return set(re.findall(r'[a-zA-Z0-9]+', text.lower()))
def keyword_score(query, chunk):
    query_terms = tokens(query) - {'how', 'should', 'a', 'when', 'using'}
    return len(query_terms & tokens(chunk['text'])) / max(1, len(query_terms))

keyword_ranked = sorted(((keyword_score(query, chunk), chunk) for chunk in chunks), reverse=True, key=lambda item: item[0])
[(round(score, 2), chunk['chunk_id'], chunk['page']) for score, chunk in keyword_ranked[:3]]

In [ ]:
# Retrieval 2: offline semantic fallback. This path never downloads a model.
topics = ['human','privacy','ethics','pedagogy','agency','assessment','safety','foundations']
terms = {
 'human':['human','learner','student','ผู้เรียน','มนุษย์'], 'privacy':['privacy','data','ข้อมูล','ความเป็นส่วนตัว'],
 'ethics':['ethic','bias','จริย','อคติ'], 'pedagogy':['school','teach','learn','โรงเรียน','การสอน','การเรียน'],
 'agency':['agency','control','accountab','ควบคุม','รับผิดชอบ'], 'assessment':['evaluate','validation','test','ประเมิน','ตรวจสอบ'],
 'safety':['safe','protect','risk','ปลอดภัย','คุ้มครอง','ความเสี่ยง'], 'foundations':['competenc','framework','สมรรถนะ','กรอบ']
}
def embed_fallback(text):
    text = text.lower()
    return [int(any(term in text for term in terms[topic])) for topic in topics]
def cosine(a, b):
    denominator = math.sqrt(sum(x*x for x in a))*math.sqrt(sum(x*x for x in b))
    return sum(x*y for x,y in zip(a,b))/denominator if denominator else 0

for chunk in chunks: chunk['embedding'] = embed_fallback(chunk['text'])
query_vector = embed_fallback(query)
ranked = sorted(((cosine(query_vector, c['embedding']), c) for c in chunks), reverse=True, key=lambda item: item[0])
[(round(score, 2), chunk['chunk_id'], chunk['text']) for score, chunk in ranked[:3]]

## Optional real ingestion / ทางเลือก: ทำ embedding จริง

Change `USE_REAL_MODEL` to `True` only when internet is available. This downloads `intfloat/multilingual-e5-small`; retain the fallback route for the live class. Compare its results with the keyword retrieval above.

In [ ]:
USE_REAL_MODEL = False  # Leave False for the guaranteed offline-safe path.
if USE_REAL_MODEL:
    !pip -q install sentence-transformers
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('intfloat/multilingual-e5-small')
    passages = [f"passage: {chunk['text']}" for chunk in chunks]
    real_vectors = model.encode(passages, normalize_embeddings=True)
    real_query = model.encode([f"query: {query}"], normalize_embeddings=True)[0]
    real_ranked = sorted(((float(real_query @ vector), chunks[i]) for i, vector in enumerate(real_vectors)), reverse=True)
    [(round(score, 2), chunk['chunk_id']) for score, chunk in real_ranked[:3]]
else:
    print('Fallback index active: no model download and no API key required.')

In [ ]:
# Turn retrieval into a citation object. Never answer beyond retrieved evidence.
best_score, best_chunk = ranked[0]
document = next(d for d in documents if d['id'] == best_chunk['document_id'])
citation = {
    'title': document['title'], 'title_th': document['title_th'], 'chunk_id': best_chunk['chunk_id'],
    'similarity': round(best_score, 2), 'source_page': document.get('page'),
    'source_url': document.get('source_url', 'See corpus/ATTRIBUTION.md')
}
pprint(citation)
assert citation['similarity'] > 0, 'The answer must not be generated without retrieved evidence.'